# Customer Shopping Behavior Analysis

### Data Cleaning, Transformation and PostgreSQL Integration Using Python

## Project Overview

This notebook is part of an end-to-end data analytics project developed using **Python, PostgreSQL, and Power BI**.

The project analyzes shopping behavior data from **3,900 customer records**, with the goal of understanding purchasing patterns and preparing structured data for further analysis and visualization.

### Objectives

- Explore the dataset and assess data quality.
- Identify and handle missing values.
- Clean and standardize the dataset.
- Perform feature engineering to support customer segmentation.
- Prepare the processed data for PostgreSQL.
- Enable further business analysis using SQL and Power BI.

### Technologies

- **Python:** Pandas and data preprocessing.
- **PostgreSQL:** Data storage and SQL analysis.
- **Power BI:** Interactive dashboard and business insights.

### Workflow

Raw Data → Data Exploration → Data Cleaning → Feature Engineering → PostgreSQL → SQL Analysis → Power BI Dashboard

In [1]:
# Import libraries
import pandas as pd
from pathlib import Path

# To import engine
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
from sqlalchemy.engine import URL

## 1. Exploratory Data Analysis (EDA)

This section examines the structure and quality of the customer shopping dataset before applying any transformations.

The initial exploration includes:

- Inspecting the first records and understanding the dataset structure.
- Reviewing data types and descriptive statistics.
- Identifying missing values and potential data quality issues.
- Examining categorical and numerical variables.

These checks help determine which preprocessing steps are necessary before loading the data into PostgreSQL.

In [2]:
# Load dataset
# Locate the dataset
project_dir = Path.cwd()

if not (project_dir / "data").is_dir():
    project_dir = project_dir.parent

# Load dataset
data_path = project_dir / "data" / "customer_shopping_behavior.csv"

df = pd.read_csv(data_path)

df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [3]:
# Information of data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [4]:
# Describe teh dataset
df.describe()

,Customer ID,Age,Purchase Amount (USD),Review Rating,Previous Purchases
count,3900.000000,3900.000000,3900.000000,3863.000000,3900.000000
mean,1950.500000,44.068462,59.764359,3.750065,25.351538
std,1125.977353,15.207589,23.685392,0.716983,14.447125
min,1.000000,18.000000,20.000000,2.500000,1.000000
25%,975.750000,31.000000,39.000000,3.100000,13.000000
50%,1950.500000,44.000000,60.000000,3.800000,25.000000
75%,2925.250000,57.000000,81.000000,4.400000,38.000000
max,3900.000000,70.000000,100.000000,5.000000,50.000000


In [5]:
# Describe teh dataset with categorical columns
df.describe(include = 'all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


## 2. Data Cleaning and Feature Engineering

The dataset is cleaned and transformed to improve consistency and prepare it for SQL-based analysis.

The preprocessing workflow includes:

- Handling missing values.
- Standardizing column names.
- Reviewing and adjusting data types where necessary.
- Creating derived variables to support customer behavior analysis.

These transformations produce a structured dataset that can be used for customer segmentation, revenue analysis, and purchasing pattern exploration.

In [6]:
# See null values
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [7]:
# Replacing null values with median
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

# See null values
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [8]:
# Transform the names of columns in better names
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')

# change the name for better name
df = df.rename(columns = {'purchase_amount_(usd)': 'purchase_amount'})

# Columns
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [9]:
# Create a columns age_group
labels = ['Young adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q = 4, labels = labels)

In [10]:
df[['age','age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young adult
2,50,Middle-aged
3,21,Young adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young adult
8,26,Young adult
9,57,Middle-aged


In [11]:
# Create a column purchase_frequency_days

frecuency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Monthly': 30,
    'Quarterly': 90,
    'Bi-Weekly': 14,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frecuency_mapping)

In [12]:
df[['purchase_frequency_days','frequency_of_purchases']].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [13]:
df[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [14]:
# To see if this columns are the same
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [15]:
# Drop the column promo_code_used
df = df.drop('promo_code_used', axis = 1)

In [16]:
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

## 3. PostgreSQL Integration

The processed dataset is integrated with a local PostgreSQL database using SQLAlchemy.

Database credentials are managed through environment variables stored in a local `.env` file, which is excluded from version control.

The database is named `customer_behavior`, and the customer records are stored in the `customer` table.

The notebook includes a connection check and an optional data-loading step. By default, uploading is disabled to avoid accidentally replacing an existing table.

Once the data is available in PostgreSQL, SQL queries can be used to investigate revenue, product performance, subscriptions, discounts, and customer segments.

In [17]:
# Locate the environment configuration
env_path = project_dir / ".env"

if not env_path.is_file():
    raise FileNotFoundError(
        "The .env file was not found in the project root."
    )

load_dotenv(dotenv_path=env_path, override=True)

# Create a conection
connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    host=os.getenv("DB_HOST", "localhost"),
    port=int(os.getenv("DB_PORT", "5432")),
    database=os.environ["DB_NAME"]
)

engine = create_engine(connection_url)

In [18]:
from sqlalchemy import text

try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT current_database(), current_user, version();")
        )
        database, user, version = result.fetchone()

        print("✅ Conexión exitosa a PostgreSQL")
        print(f"Base de datos: {database}")
        print(f"Usuario: {user}")
        print(f"Versión: {version.split(',')[0]}")

except Exception as e:
    print(f"❌ Error de conexión: {e}")

✅ Conexión exitosa a PostgreSQL
Base de datos: customer_behavior
Usuario: postgres
Versión: PostgreSQL 18.6 on x86_64-windows


In [19]:
# Verify PostgreSQL table
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM customer")
    )
    total_rows = result.scalar()

print("PostgreSQL connection successful.")
print(f"Total records in customer table: {total_rows}")

PostgreSQL connection successful.
Total records in customer table: 3900


### Loading Data into PostgreSQL

The cleaned dataset can be loaded into PostgreSQL using SQLAlchemy and Pandas.

The database used in this project is named `customer_behavior`, and the destination table is `customer`.

The dataset was already imported during the development of this project. To reproduce the database setup, users can execute the following code after configuring their PostgreSQL credentials.

**Note:** The `replace` option overwrites an existing table. It should only be used when recreating the dataset intentionally.

In [20]:
# Optional: Upload cleaned data to PostgreSQL
# Set to True only when you want to recreate the table.

UPLOAD_TO_POSTGRESQL = False

if UPLOAD_TO_POSTGRESQL:
    df.to_sql(
        name="customer",
        con=engine,
        if_exists="replace",
        index=False
    )
    print("Dataset uploaded successfully.")
else:
    print("Upload skipped. Existing table preserved.")

Upload skipped. Existing table preserved.


## 4. Conclusion

This notebook demonstrates the data preparation stage of an end-to-end customer shopping behavior analysis project.

Using Python, the dataset was explored, cleaned, and transformed into a structured format suitable for relational database storage.

PostgreSQL is used for further analytical queries, while Power BI provides interactive visualizations to communicate business insights.

The complete project, including SQL queries and the Power BI dashboard, is available in the corresponding GitHub repository.

In [21]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("""
        SELECT
            current_database(),
            current_user,
            inet_server_addr(),
            inet_server_port();
    """))

    print(result.fetchone())

('customer_behavior', 'postgres', '::1', 5432)
